# Topic 6 그림 코드 모음 — 강의용 Colab 데모

**확률통계 · Topic 6 · Continuous Random Variables and Sampling**

슬라이드 [T06_slides_v2.md](slides/T06_slides_v2.md) 에 쓴 그림 3개를 만드는 코드를 모았다.
슬라이드는 고정된 PNG지만, 이 노트북은 **강의 중 숫자를 바꿔가며 즉석에서 다시 그려볼 수 있다.**

| 그림 | 슬라이드 | 원본 스크립트 |
|---|---|---|
| ① 막대를 좁혀도 세로축 값이 그대로 | 6쪽 `[S]` 20만 명의 키를 히스토그램으로 | `figs_src_v2/t06v2_hist_to_pdf.py` |
| ② PDF의 면적 = CDF의 높이 차 | 12쪽 `[C]` 면적과 높이 차는 같은 값이다 | `figs_src_v2/t06v2_pdf_area.py` |
| ③ inverse transform sampling | 22쪽 `[C]` CDF를 거꾸로 탄다 | `figs_src_v2/t06v2_inverse_transform.py` |

⚠️ 실제 슬라이드 그림은 Windows 로컬에서 `Malgun Gothic`으로 렌더했다. 이 노트북은 Colab(Linux)이라
나눔고딕을 대신 설치해서 쓴다 — 글꼴만 다르고 배치·색·수치는 슬라이드와 동일하다.

**맨 위 설정 셀을 한 번 실행한 뒤, 그림 셀은 순서와 상관없이 원하는 것만 실행하면 된다.**

In [ ]:
# 설정: 한글 글꼴 + 스타일 (맨 처음 한 번만 실행)
# Colab은 Linux라 한글 글꼴이 기본으로 없다. 나눔고딕을 설치해 등록한다.
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

try:
    for f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(f)
    KOREAN_FONT = "NanumGothic"
except Exception:
    KOREAN_FONT = "DejaVu Sans"  # 설치 실패 시 - 한글은 깨지지만 그림은 그려진다
    print("나눔고딕 설치/등록 실패 - DejaVu Sans로 대신한다 (한글이 깨질 수 있음)")

# 슬라이드 테마와 같은 색 (assets/deckstyle.py 와 동일)
C = {
    "accent": "#3b4fd8", "teal": "#0d9488", "orange": "#d97d17",
    "purple": "#8b5cf6", "pink": "#d9457f", "ink": "#0f172a",
    "body": "#4b5768", "muted": "#97a3b6", "line": "#e6eaf1", "soft": "#f7f9fc",
}
CYCLE = [C["accent"], C["teal"], C["orange"], C["purple"], C["pink"], C["muted"]]

mpl.rcParams.update({
    "font.family": KOREAN_FONT,
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.5,
    "ytick.labelsize": 12.5,
    "legend.fontsize": 12.5,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": C["line"],
    "grid.linewidth": 1.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#d7dce6",
    "figure.dpi": 110,
})

## ① 막대를 좁혀도 세로축 값이 그대로 — 밀도라는 관찰

핵심은 "곡선이 된다"가 아니다. **막대를 40배 좁혔는데 평균 근처 높이가 거의 그대로**라는
관찰이다. 그래서 세로축은 확률이 아니라 **밀도**다. `WIDTHS` 를 바꿔가며 다시 확인해볼 수 있다.

⚠️ 반드시 `density=True` 로 그릴 것. 빈도로 그리면 막대가 좁아질수록 높이가 낮아져
정반대 인상을 준다.

In [ ]:
N = 200_000
MU, SD = 170.0, 6.0
rng = np.random.default_rng(20260302)          # 개강일 시드 (전 Topic 공통)
h = rng.normal(MU, SD, N)

WIDTHS = [8.0, 3.0, 1.0, 0.2]
COLS = [C["orange"], C["teal"], C["accent"], C["purple"]]

fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.1), sharey=True)

peak_at_mu = []
for ax, w, col in zip(axes, WIDTHS, COLS):
    edges = np.arange(150, 190 + w, w)
    dens, _, _ = ax.hist(h, bins=edges, density=True, color=col, alpha=0.85,
                         edgecolor="white", linewidth=0.5)
    # 평균 근처 막대의 높이 (밀도)
    k = np.searchsorted(edges, MU) - 1
    peak_at_mu.append(dens[k])
    ax.set_title(f"막대 폭 {w:g} cm", color=C["ink"], fontsize=14)
    ax.set_xlim(150, 190)
    ax.set_xlabel("키 (cm)")

# 마지막 패널에만 이론 곡선을 겹친다
xs = np.linspace(150, 190, 400)
pdf = np.exp(-((xs - MU) ** 2) / (2 * SD**2)) / (SD * np.sqrt(2 * np.pi))
axes[-1].plot(xs, pdf, color=C["ink"], linewidth=2.2)
axes[-1].text(151, pdf.max() * 0.92, "이론 곡선", color=C["ink"],
              fontsize=13, fontweight="bold")

axes[0].set_ylabel("세로축 값")
for w, p in zip(WIDTHS, peak_at_mu):
    print(f"막대 폭 {w:>4}cm  평균 근처 밀도 {p:.4f}")
print(f"이론 최대 밀도 {1 / (SD * np.sqrt(2 * np.pi)):.4f}")
print(f"표본 {N:,}개 · 폭을 {WIDTHS[0] / WIDTHS[-1]:.0f}배 좁혀도 높이는 그대로")

fig.subplots_adjust(wspace=0.12)
plt.show()

## ② PDF의 면적 = CDF의 높이 차

왼쪽 파란 면적과 오른쪽 두 점의 높이 차가 **같은 값**이라는 것을 눈으로 확인한다.
Exponential(0.5) 에서 $P[1 \le X \le 3]$ 을 쓴다. `A`, `B` 를 바꿔 다른 구간으로도
같은 확인을 해볼 수 있다.

In [ ]:
LAM = 0.5
A, B = 1.0, 3.0

x = np.linspace(0, 9, 600)
pdf = LAM * np.exp(-LAM * x)
cdf = 1 - np.exp(-LAM * x)

FA, FB = 1 - np.exp(-LAM * A), 1 - np.exp(-LAM * B)
area = FB - FA
print(f"F({A}) = {FA:.4f}   F({B}) = {FB:.4f}")
print(f"P[{A} <= X <= {B}] = {area:.4f}   (= e^-0.5 - e^-1.5)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.5))

# ── 왼쪽: PDF 아래 면적 ────────────────────────────────────
ax1.plot(x, pdf, color=C["accent"], linewidth=2.4)
m = (x >= A) & (x <= B)
ax1.fill_between(x[m], 0, pdf[m], color=C["accent"], alpha=0.28)
ax1.text((A + B) / 2, 0.10, f"넓이\\n{area:.4f}", ha="center", va="center",
         fontsize=14, fontweight="bold", color=C["ink"])
ax1.set_title("PDF — 넓이가 확률이다", color=C["ink"])
ax1.set_xlabel("x")
ax1.set_ylabel("$f_X(x)$  밀도")
ax1.set_xlim(0, 9)
ax1.set_ylim(0, 0.55)
ax1.set_xticks([0, A, B, 6, 9])

# ── 오른쪽: CDF 높이 차 ───────────────────────────────────
ax2.plot(x, cdf, color=C["teal"], linewidth=2.4)
for v, F, name in [(A, FA, f"$F({A:g})$ = {FA:.4f}"), (B, FB, f"$F({B:g})$ = {FB:.4f}")]:
    ax2.plot([v, v], [0, F], color=C["muted"], linewidth=1.2, linestyle=":")
    ax2.plot(v, F, "o", color=C["teal"], markersize=8, zorder=5)
    ax2.text(v + 0.25, F - 0.055, name, fontsize=12.5, color=C["ink"])

# 높이 차를 화살표로
ax2.annotate("", xy=(8.2, FB), xytext=(8.2, FA),
             arrowprops=dict(arrowstyle="<|-|>", color=C["orange"], linewidth=2.0))
ax2.text(7.9, (FA + FB) / 2, f"차이\\n{area:.4f}", ha="right", va="center",
         fontsize=13.5, fontweight="bold", color=C["orange"])

ax2.set_title("CDF — 높이 차가 같은 값", color=C["ink"])
ax2.set_xlabel("x")
ax2.set_ylabel("$F_X(x)$  누적확률")
ax2.set_xlim(0, 9)
ax2.set_ylim(0, 1.08)
ax2.set_xticks([0, A, B, 6, 9])

plt.tight_layout()
plt.show()

## ③ inverse transform sampling — 균등난수 하나로 아무 분포나 만든다

왼쪽: 세로축(0~1)에서 균등하게 뽑아 CDF 곡선을 타고 내려오면 가로축 표본이 된다.
오른쪽: 그렇게 만든 20,000개 표본의 히스토그램이 이론 PDF와 겹친다.

⚠️ inverse transform sampling 은 Chan 교재에 없다 (CS109 + 직접 작성).

In [ ]:
LAM = 0.5
N = 20_000
rng = np.random.default_rng(20260302)

u = rng.random(N)
xs_sample = -np.log(1 - u) / LAM              # F^{-1}(u)
print(f"표본 {N:,}개 · 평균 {xs_sample.mean():.4f} (이론 {1 / LAM:.4f})")
print(f"중앙값 {np.median(xs_sample):.4f} (이론 {np.log(2) / LAM:.4f})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 3.5))

# ── 왼쪽: CDF를 거꾸로 타는 원리 ───────────────────────────
x = np.linspace(0, 9, 500)
cdf = 1 - np.exp(-LAM * x)
ax1.plot(x, cdf, color=C["teal"], linewidth=2.4)

for uu, col in [(0.25, C["accent"]), (0.60, C["orange"]), (0.90, C["purple"])]:
    xx = -np.log(1 - uu) / LAM
    ax1.annotate("", xy=(xx, uu), xytext=(0, uu),
                 arrowprops=dict(arrowstyle="-|>", color=col, linewidth=1.7))
    ax1.annotate("", xy=(xx, 0), xytext=(xx, uu),
                 arrowprops=dict(arrowstyle="-|>", color=col, linewidth=1.7))
    ax1.text(-0.15, uu, f"u={uu:.2f}", ha="right", va="center",
             fontsize=12, fontweight="bold", color=col)
    ax1.text(xx, -0.09, f"{xx:.2f}", ha="center", va="top",
             fontsize=12, fontweight="bold", color=col)

ax1.set_title("① 세로축에서 뽑아 ② 곡선을 타고 ③ 가로축으로", color=C["ink"], fontsize=13.5)
ax1.set_xlabel("x  =  $F^{-1}(u)$")
ax1.set_ylabel("u ~ Uniform(0, 1)")
ax1.set_xlim(-1.6, 9)
ax1.set_ylim(-0.16, 1.06)

# ── 오른쪽: 만들어진 표본 ──────────────────────────────────
ax2.hist(xs_sample, bins=60, range=(0, 12), density=True, color=C["accent"],
         alpha=0.75, edgecolor="white", linewidth=0.4, label="만든 표본 20,000개")
xx = np.linspace(0, 12, 400)
ax2.plot(xx, LAM * np.exp(-LAM * xx), color=C["ink"], linewidth=2.2,
         label="이론 PDF  $\\lambda e^{-\\lambda x}$")
ax2.set_title("균등난수만 썼는데 지수분포가 나온다", color=C["ink"], fontsize=13.5)
ax2.set_xlabel("x")
ax2.set_ylabel("밀도")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

---
이 노트북은 채점 대상이 아니다. 슬라이드 그림을 재생성하는 [figs_src_v2/](figs_src_v2/) 스크립트가
원본이며, 슬라이드 PNG를 바꾸려면 그쪽을 고치고 다시 실행해야 한다 — 이 노트북은 강의 중
라이브 데모·질의응답용 사본이다. Exponential 샘플러를 직접 만드는 실습은
[lab/T06_lab.ipynb](lab/T06_lab.ipynb) 에 있다.